
# 05 — ML pipeline: baseline vs. scaling versions

Four runs over one matched frame: two CV setups crossed with two column sets.

| | `_Ever` in | `_Ever` out |
|---|---|---|
| **baseline version** —  no feature scaling | `baseline_with_ever` | `baseline_no_ever` |
| **scaling version** — conventional parameters, `StandardScaler` in a `Pipeline` for the linear models | `scaling_with_ever` | `scaling_no_ever` |

Both versions split cross-validation folds **by matched stratum, never by row**, so a case
and its matched control always land in the same fold.

Both versions build folds the same way — `np.unique(strata)` fed to
`KFold(n_splits=10, shuffle=True, random_state=12345)` — so fold membership is identical
across all four runs and the deltas are comparable in every direction.

Run order: restart and run all. Strip outputs (`nbstripout`) before committing.

## Setup

In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier)
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, roc_auc_score, accuracy_score,
                             recall_score)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAS_XGB = False

def boost(seed):
    if HAS_XGB:
        return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                             subsample=0.8, colsample_bytree=0.8,
                             eval_metric="logloss", random_state=seed, n_jobs=-1)
    return HistGradientBoostingClassifier(random_state=seed)
    
SEED     = 12345
N_SPLITS = 10

In [ ]:

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data"

SOURCE_FILE = DATA_DIR / "osteo_features.csv"
OHE_FILE    = DATA_DIR / "05_data_OHE.csv" #to export later
CONFIG_FILE = Path("config_ml.yaml")     # relative to the notebook's cwd, as before

In [ ]:

one = pd.read_csv(SOURCE_FILE, low_memory=False)
print("source:", one.shape)


## Column groups from `config_ml.yaml`

- `config_ml.yaml` is the config names of the study's variables.
- For the 2 lab groups in this config file (`calcium_sodium`, `ever_worst`), the models are not fitted on the study variable itself — such as `Calcium_Avg_Ever` — but on _two columns engineered from it_: `Calcium_Avg_Ever_decile` (the binned value) and `Calcium_Avg_Ever_measured`(whether the lab was drawn at all).

So two dicts:

- `GROUPS_RAW` — study variable names, as the config writes them. The `_Ever` section below matches on these.
- `GROUPS` — _engineered column names_, what the models actually see.

In [ ]:

cfg = yaml.safe_load(open(CONFIG_FILE))

KEYS       = cfg["keys"]
GROUPS_RAW = {name: g["columns"] for name, g in cfg["groups"].items()}

for name, cols in GROUPS_RAW.items():
    print(f"{name}: {cols}")

In [ ]:

GROUPS = dict(GROUPS_RAW)

LAB_COLS = GROUPS.pop("calcium_sodium") + GROUPS.pop("ever_worst")

GROUPS["labs_decile"]   = [c + "_decile"   for c in LAB_COLS]
GROUPS["labs_measured"] = [c + "_measured" for c in LAB_COLS]

In [ ]:

# reverse lookup: frame column -> config group, dummy names included
COL_TO_GROUP = {c: g for g, cols in GROUPS.items() for c in cols}

def group_of(col):
    if col in COL_TO_GROUP:
        return COL_TO_GROUP[col]
    for base, g in COL_TO_GROUP.items():        # "sex_F" -> "sex" -> demographics
        if col.startswith(base + "_"):
            return g
    return "unknown"

## One-hot encoding and pair integrity

In [ ]:

ALL_VARS = [c for cols in GROUPS.values() for c in cols]
study_vars    = one[KEYS + ALL_VARS]

numeric_subset     = study_vars.select_dtypes("number")
categorical_subset = pd.get_dummies(study_vars[GROUPS["demographics"]], dtype=int)

data_OHE   = pd.concat([numeric_subset, categorical_subset], axis=1)
DEMO_COLS  = list(categorical_subset.columns)
MODEL_COLS = [c for c in data_OHE.columns if c not in KEYS]

In [ ]:

# keep only complete rows, then only strata that are still a clean 1-case / 1-control pair
data_OHE = data_OHE.dropna()

pair     = data_OHE.groupby("Strata")["osteo_label"]
data_OHE = data_OHE[pair.transform("size").eq(2) & pair.transform("sum").eq(1)]
data_OHE = data_OHE.reset_index(drop=True)

print("frame        :", data_OHE.shape)
print("strata       :", data_OHE["Strata"].nunique())
print("model columns:", len(MODEL_COLS))
print("demo dummies :", len(DEMO_COLS))
print("by group     :", {name: sum(group_of(c) == name for c in MODEL_COLS)
                         for name in GROUPS})

assert len(data_OHE) == 2 * data_OHE["Strata"].nunique()

In [ ]:

# row-level file, gitignored — nothing downstream of here is committed
data_OHE.to_csv(OHE_FILE, index=False)

In [ ]:
MODEL_COLS


## Cross-validation folds — split on strata, not on rows

`uniq` and `fold` below are built once and used by **both** versions, so all four runs
see the same 10 folds.

In [ ]:

y      = data_OHE["osteo_label"].to_numpy()
strata = data_OHE["Strata"].to_numpy()

uniq = np.unique(strata)
fold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

print("class counts:", np.unique(y, return_counts=True))
print(f"{len(uniq)} strata, {N_SPLITS} folds")

In [ ]:

# verify no pair straddles a fold boundary before fitting anything
for tr_u, te_u in fold.split(uniq):
    tr = np.isin(strata, uniq[tr_u])
    te = np.isin(strata, uniq[te_u])
    assert not (set(strata[tr]) & set(strata[te]))
    assert tr.sum() % 2 == 0 and te.sum() % 2 == 0

print("pairs intact")


## Column sets: with and without `_Ever`

`_Ever` columns span the full record, including the period after the index date.
`_Prior` columns are window-restricted. Everything else — frame, folds, seed, estimator
settings — is held fixed between the two column sets, so the AUC delta is attributable
to these six columns.

In [ ]:
# name rule
EVER_COLS = []
for col in MODEL_COLS:
    if "_Ever" in col:
        EVER_COLS.append(col)

# cross-check against config: GROUPS_RAW["ever_worst"] holds the base names,
# startswith() picks up the _decile / _measured children built from each base
config_ever = []
for base in GROUPS_RAW["ever_worst"]:
    for col in MODEL_COLS:
        if col.startswith(base):
            config_ever.append(col)

assert set(EVER_COLS) == set(config_ever), "name rule and config disagree"

In [ ]:
NO_EVER_COLS = []
for col in MODEL_COLS:
    if col not in EVER_COLS:
        NO_EVER_COLS.append(col)

assert len(NO_EVER_COLS) + len(EVER_COLS) == len(MODEL_COLS)

print(f"MODEL_COLS   : {len(MODEL_COLS)}")
print(f"_Ever matched: {len(EVER_COLS)}")
for col in EVER_COLS:
    print("     ", col)
print(f"NO_EVER_COLS : {len(NO_EVER_COLS)}")


## Baseline version — no feature scaling


In [ ]:
def make_model(name, version, seed=SEED):
    """One factory. `name` picks the family, `version` picks the settings."""

    if name == 'LR':
        if version == 'baseline':
            return LogisticRegression(solver='liblinear', random_state=seed)
        return Pipeline([("sc", StandardScaler()),
                         ("clf", LogisticRegression(solver="liblinear",
                                                    random_state=seed))])

    if name == 'RF':
        return RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                      n_jobs=-1, random_state=seed)

    if name == 'SVM':
        svm = LinearSVC(C=1.0, dual="auto", max_iter=5000, random_state=seed, loss='hinge')
        if version == 'baseline':
            return CalibratedClassifierCV(svm)
        return Pipeline([("sc", StandardScaler()),
                         ("clf", CalibratedClassifierCV(svm))])

    if name == 'XGB':
        return boost(seed)          # identical in both versions

    if name == 'AdaBoost':
        if version != 'baseline':
            raise ValueError("AdaBoost is baseline-only")
        base = LogisticRegression(solver='lbfgs', random_state=seed)
        return AdaBoostClassifier(n_estimators=100, estimator=base,
                                  learning_rate=1, random_state=seed)

    raise ValueError(f"unknown model: {name}")

In [ ]:
def TRAIN_MODEL_ML(data_ohe, classifier, feature_cols, version,
                   n_splits=N_SPLITS, seed=SEED):

    data = data_ohe.reset_index(drop=True).copy()

    X = data[["Strata"] + feature_cols]
    Y = data[["Strata", "osteo_label"]]

    inner_fold   = KFold(n_splits, shuffle=True, random_state=seed)
    inner_strata = np.unique(data["Strata"])

    all_preds   = np.full(len(data), -1)
    probability = np.full(len(data), np.nan)

    training_acc = []
    testing_acc  = []

    for train_index, test_index in inner_fold.split(inner_strata):

        train_strata = inner_strata[train_index]
        test_strata  = inner_strata[test_index]

        X_train = X.loc[X["Strata"].isin(train_strata)].drop(["Strata"], axis=1)
        X_test  = X.loc[X["Strata"].isin(test_strata)].drop(["Strata"], axis=1)

        y_train = Y.loc[Y["Strata"].isin(train_strata)]["osteo_label"]
        y_test  = Y.loc[Y["Strata"].isin(test_strata)]["osteo_label"]

        clf = make_model(classifier, version)
        fit = clf.fit(X_train, y_train)

        all_preds[y_test.index]   = clf.predict(X_test)
        probability[y_test.index] = clf.predict_proba(X_test)[:, 1]

        training_acc.append(fit.score(X_train, y_train))
        testing_acc.append(fit.score(X_test, y_test))

    return all_preds, probability, training_acc, testing_acc

In [ ]:
def RUN_VERSION(data_ohe, classifiers, feature_cols, version):
    """One fold loop, both versions. Only the estimator settings differ.
    verion = scaling or not"""
    rows = []

    for classifier in classifiers:
        oof_pred, oof_prob, train_acc, test_acc = TRAIN_MODEL_ML(
            data_ohe, classifier, feature_cols, version
        )

        fpr, tpr, threshold = roc_curve(y, oof_prob)

        rows.append({
            'classifier':   classifier,
            'FPR':          fpr,
            'TPR':          tpr,
            'AUC':          roc_auc_score(y, oof_prob),
            'training_acc': train_acc,
            'testing_acc':  test_acc,
            'y_pred':       oof_pred,
            'y_prob':       oof_prob,
        })

    out = pd.DataFrame(rows).set_index('classifier')
    return out


## Scaling version — conventional parameters, scaled linear models

`cv_by_strata` reads `data_OHE`, `y`, `strata`, `uniq` and `fold` from the module level.
They are fixed for the whole notebook; only `feature_cols` varies between runs.

## Run all four

In [ ]:
BASELINE_MODELS = ['LR', 'RF', 'SVM', 'XGB', 'AdaBoost']
SCALING_MODELS  = ['LR', 'RF', 'SVM', 'XGB']


baseline_with_ever = RUN_VERSION(data_OHE, BASELINE_MODELS, MODEL_COLS,   'baseline')
print("baseline_with_ever done")

In [ ]:
baseline_no_ever   = RUN_VERSION(data_OHE, BASELINE_MODELS, NO_EVER_COLS, 'baseline')

print("baseline_no_ever done")

In [ ]:

scaling_with_ever  = RUN_VERSION(data_OHE, SCALING_MODELS, MODEL_COLS,   'scaling')
print("scaling_with_ever done")

In [ ]:

scaling_no_ever    = RUN_VERSION(data_OHE, SCALING_MODELS, NO_EVER_COLS, 'scaling')
print("scaling_no_ever done")

In [ ]:

RUNS = {
    ("baseline", "with_ever"): baseline_with_ever,
    ("baseline", "no_ever"):   baseline_no_ever,
    ("scaling",  "with_ever"): scaling_with_ever,
    ("scaling",  "no_ever"):   scaling_no_ever,
}

In [ ]:
print(round(baseline_with_ever.loc["XGB", "AUC"], 6))
print(round(scaling_with_ever.loc["XGB", "AUC"], 6))

## Reports

In [ ]:

def REPORT(run_df, label):
    print(f"########## {label} ##########")
    for name in run_df.index:
        row = run_df.loc[name]
        print(f"\n===== {name} =====")
        print(classification_report(y, row["y_pred"]))
        print(confusion_matrix(y, row["y_pred"]))
        print("AUC           :", round(row["AUC"], 4))
        print("mean train acc:", round(np.mean(row["training_acc"]), 4))
        print("mean test acc :", round(np.mean(row["testing_acc"]), 4))
    print()


for (version, colset), run_df in RUNS.items():
    REPORT(run_df, f"{version} / {colset}")


### AUC table

Long-to-wide on `colset` only. The two versions are **not** pivoted against each other,
because `XGB` names a different estimator in each and `LR`/`LogReg`, `SVM_linear`/`SVM`
are different parameterisations — aligning them into shared rows would imply a
like-for-like comparison that isn't there.

In [ ]:

rows = []
for (version, colset), run_df in RUNS.items():
    for model_name in run_df.index:
        rows.append({"version": version,
                     "colset":  colset,
                     "model":   model_name,
                     "AUC":     run_df.loc[model_name]["AUC"]})

all_runs = pd.DataFrame(rows)

auc_table = all_runs.pivot(index=["version", "model"], columns="colset", values="AUC")
auc_table = auc_table[["with_ever", "no_ever"]]
auc_table["delta"] = auc_table["no_ever"] - auc_table["with_ever"]

print(auc_table.round(4))


### `_Ever` delta, with recall split by class

AUC alone hides where the loss lands. Both drivers now return out-of-fold predictions,
so case and control recall are available for all four runs.

In [ ]:

def DELTA_TABLE(with_ever_df, no_ever_df, version):
    rows = []

    for name in with_ever_df.index:
        w = with_ever_df.loc[name]
        n = no_ever_df.loc[name]

        rows.append({
            "version":               version,
            "model":                 name,
            "auc_with_ever":         w["AUC"],
            "auc_no_ever":           n["AUC"],
            "auc_delta":             n["AUC"] - w["AUC"],
            "case_recall_with_ever": recall_score(y, w["y_pred"], pos_label=1),
            "case_recall_no_ever":   recall_score(y, n["y_pred"], pos_label=1),
            "ctrl_recall_with_ever": recall_score(y, w["y_pred"], pos_label=0),
            "ctrl_recall_no_ever":   recall_score(y, n["y_pred"], pos_label=0),
        })

    return pd.DataFrame(rows).set_index(["version", "model"])


delta = pd.concat([
    DELTA_TABLE(baseline_with_ever, baseline_no_ever, "baseline"),
    DELTA_TABLE(scaling_with_ever,  scaling_no_ever,  "scaling"),
])

print(delta.round(4))

### ROC curves

In [ ]:

def PLOT_MULTIPLE_ROC(result_df, title="ROC Curve Comparison", savepath=None):
    fig = plt.figure(figsize=(8, 6))

    for i in result_df.index:
        plt.plot(result_df.loc[i]["FPR"],
                 result_df.loc[i]["TPR"],
                 label="{}, AUC={:.3f}".format(i, result_df.loc[i]["AUC"]))

    plt.plot([0, 1], [0, 1], color="orange", linestyle="--")
    plt.xticks(np.arange(0.0, 1.1, step=0.1))
    plt.xlabel("False Positive Rate", fontsize=15)
    plt.yticks(np.arange(0.0, 1.1, step=0.1))
    plt.ylabel("True Positive Rate", fontsize=15)
    plt.title(title, fontweight="bold", fontsize=15)
    plt.legend(prop={"size": 11}, loc="lower right")

    if savepath:
        fig.savefig(savepath, dpi=150, bbox_inches="tight")

    plt.show()


def LABELLED(version):
    """Stack the two column sets of one version into a single plotting frame."""
    pieces = []
    for (v, colset), run_df in RUNS.items():
        if v != version:
            continue
        piece = run_df.copy()
        piece.index = [f"{colset} / {m}" for m in piece.index]
        pieces.append(piece)
    return pd.concat(pieces)

In [ ]:

PLOT_MULTIPLE_ROC(LABELLED("baseline"), title="Baseline version — _Ever in vs. out")
PLOT_MULTIPLE_ROC(LABELLED("scaling"),  title="Scaling version — _Ever in vs. out")

In [ ]:
PLOT_MULTIPLE_ROC(
    scaling_no_ever,
    title="Variable Scaling version, _Ever excluded — ROC by model",
    savepath=PROJECT_ROOT / "outputs" / "figures" / "roc_scaling_no_ever.png",
)

## Baseline vs. scaling — what actually differs

Same frame, same 10 folds, same seed (`12345`). `TRAIN_MODEL_ML` builds folds from
`np.unique(strata)` in both versions, so fold membership is identical and the two
versions are directly comparable.

`make_model` is the only place the versions diverge:

| model | baseline | scaling |
|---|---|---|
| `LR` | `LogisticRegression(solver='liblinear')` | same estimator inside a `Pipeline` with `StandardScaler` |
| `SVM` | `CalibratedClassifierCV(LinearSVC(...))` | same, inside a `Pipeline` with `StandardScaler` |
| `RF` | `n_estimators=300, min_samples_leaf=5` | identical |
| `XGB` | `boost(seed)` | identical |
| `AdaBoost` | baseline only | — |

`RF` and `XGB` are scale-invariant and are passed through unchanged, so their two rows
in the AUC table are the same fit twice — a check that the fold machinery is
deterministic, not a comparison. Only `LR` and `SVM` carry any version signal.

`AdaBoost` has no scaling counterpart and does not appear in the version comparison.

### Convergence warnings

`SVM` uses `loss='hinge'` with `dual="auto"` and `max_iter=5000`, which does not converge
on this frame in either version. The warnings are expected. Because the solver stops
early, the small `SVM` differences between versions (third decimal, and in opposite
directions across the two column sets) are solver noise rather than a scaling effect.


## Conclusion

**Scaling vs baseline**: By scaling, I mean applied standardScaler to variables. Tree-based models wont be affected by variable scaling, it's scaling-invariant, while LR and SVM was affect. Based on the results we have, scaling effects here is very small. LR, RF, and XGBoost match to four decimals across versions. Only SVM moves at all, and only in the third decimal — noise from a non-converged solver. Scaling can be dropped as a variable of interest.

**With `_Ever` vs without**: ~0.059 AUC. The drop is uniform across every model family (0.056–0.067), so it reflects signal in the data, not a quirk of any one algorithm. Because _Ever columns span the full record including post-diagnosis time, that signal wouldn't be available at prediction time. The 0.059 is the size of the leak.

The **recall columns** show where the leak was working: dropping _Ever costs 0.08–0.12 in case recall while control recall stays flat or improves.

### What this does not settle

- The demographic dummies (`sex_*`, `Race_Cat_combine_*`, `Age_combine_*`) are the
  matching variables and are still in the feature set in all four runs. No drop-column test here — the notebook never refits without them, so it has no number for what they contribute. Meaning I can't say from this notebook whether the matching variables help, hurt, or do nothing — I only have the version where they're in.
- `_Ever` is one candidate column family. `Calcium_Closest_Osteo_*` and
  `Sodium_Closest_Osteo_*` are index-anchored and survive into `NO_EVER_COLS`; they are
  not tested by this notebook.
- No pair-level metric is computed. Every number above is a row-level metric on a
  matched design.